<a href="https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishpree1t7/flyrank_work/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I chose a logistic regression model because the task is binary classification: predicting whether a content item is declining or not declining.

Logistic regression is appropriate because it is simple, interpretable, and provides a useful comparison against the Week-4 rule-based baseline. The goal is not to maximize complexity, but to test whether a learned model can improve the baseline while keeping the reasoning understandable.

I will use the same target definition as the baseline: `is_declining_label = 1` when `trend_direction` is `down`.

The model will be evaluated using the same decision-oriented metric used in Week 4, so the comparison remains fair.

In [25]:
!git clone https://github.com/ishpree1t7/flyrank_work.git

fatal: destination path 'flyrank_work' already exists and is not an empty directory.


In [26]:
import os

print(os.listdir("/content/flyrank_work"))

['README.md', 'work', 'notebooks', 'requirements.txt', '.github', 'skills', 'LICENSE', 'outputs', 'docs', 'submission', 'SETUP.md', 'scripts', '.git', 'data', 'DATA_USE.md', 'AGENTS.md', '.gitignore', 'CLAUDE.md', 'GUIDE.md']


In [27]:
!find /content/flyrank_work -maxdepth 4 -type f | head -50

/content/flyrank_work/README.md
/content/flyrank_work/work/capstone_report_template.md
/content/flyrank_work/work/README.md
/content/flyrank_work/work/notebooks/w07_action_playbook.ipynb
/content/flyrank_work/work/notebooks/capstone.ipynb
/content/flyrank_work/work/notebooks/w05_model.ipynb
/content/flyrank_work/work/notebooks/w06_validation_audit.ipynb
/content/flyrank_work/work/notebooks/w03_feature_leakage_check.ipynb
/content/flyrank_work/work/notebooks/w04_baseline_score.ipynb
/content/flyrank_work/work/notebooks/w01_research_question.ipynb
/content/flyrank_work/work/notebooks/w04_signal_audit.ipynb
/content/flyrank_work/work/notebooks/w02_ml_task_framing.ipynb
/content/flyrank_work/work/notebooks/w03_data_contract.ipynb
/content/flyrank_work/notebooks/02_your_first_readable_model.ipynb
/content/flyrank_work/notebooks/01_first_look_and_discovery.ipynb
/content/flyrank_work/notebooks/03_working_with_the_full_release.ipynb
/content/flyrank_work/requirements.txt
/content/flyrank_work

In [28]:
from pathlib import Path

w04 = Path("/content/flyrank_work/work/notebooks/w04_baseline_score.ipynb")

print(w04.exists())
print(w04)

True
/content/flyrank_work/work/notebooks/w04_baseline_score.ipynb


In [29]:
import json

with open(w04, "r", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if "DATA_PATH" in source or "read_csv" in source or "read_parquet" in source:
        print(f"\n--- CELL {i} ---")
        print(source)


--- CELL 6 ---
import pandas as pd

DATA_PATH = "flyrank_work/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

--- CELL 17 ---
import pandas as pd
import numpy as np
from pathlib import Path

# Load the starter dataset
DATA_PATH = Path("flyrank_work/data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

# Evaluation-only target.
# NEVER use this column in the score.
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))


In [30]:
from pathlib import Path

# Check whether the dataset exists anywhere inside the cloned repo
matches = list(Path("/content/flyrank_work").rglob("content_refresh_anonymized.csv"))

print("Matches:", len(matches))
for p in matches:
    print(p)

Matches: 1
/content/flyrank_work/data/raw/content_refresh_anonymized.csv


In [31]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/flyrank_work/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [32]:
import pandas as pd
import numpy as np

# Create the evaluation target
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Shape: (30000, 45)
Declining rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a stratified random train/test split with 80% of the data for training and 20% for testing.

A stratified split keeps the proportion of declining and non-declining content similar in both sets. This makes the comparison more stable because the target distribution is preserved.

I use a fixed random state so the split is reproducible. The test set is kept separate from model training and is used only for final evaluation.

In [33]:
from sklearn.model_selection import train_test_split

# Features allowed for the model
features = [
    "days_since_last_update",
    "impressions_90d"
]

X = df[features].copy()
y = df["is_declining_label"].copy()

# Keep only rows with complete model inputs
data = pd.concat([X, y], axis=1).dropna()

X = data[features]
y = data["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train decline rate:", round(y_train.mean(), 3))
print("Test decline rate:", round(y_test.mean(), 3))

Train shape: (24000, 2)
Test shape: (6000, 2)
Train decline rate: 0.542
Test decline rate: 0.542


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I train a logistic regression model using the same two inputs used by my Week-4 baseline: `days_since_last_update` and `impressions_90d`.

The model is evaluated on the same held-out test set. I compare Precision@10, Precision@20, and Precision@50 with the Week-4 baseline so that the comparison uses the same decision-oriented metrics.

The purpose is to test whether a simple learned model improves ranking quality over the hand-built baseline.

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score

# Train logistic regression
model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=1000))
])

model.fit(X_train, y_train)

# Score the held-out test set
test_scores = model.predict_proba(X_test)[:, 1]


def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.iloc[order].mean()


for k in [10, 20, 50]:
    print(f"Precision@{k}: {precision_at_k(y_test.reset_index(drop=True), test_scores, k):.3f}")

# Week-4 baseline score
# Same two inputs, using the hand-built baseline idea:
# prioritize stale content with meaningful impression volume.

baseline_scores = (
    df.loc[X_test.index, "days_since_last_update"].fillna(0)
    * np.log1p(df.loc[X_test.index, "impressions_90d"].fillna(0))
)

y_test_baseline = y_test.copy()

for k in [10, 20, 50]:
    score = precision_at_k(
        y_test_baseline.reset_index(drop=True),
        baseline_scores.to_numpy(),
        k
    )
    print(f"Baseline Precision@{k}: {score:.3f}")

Precision@10: 0.400
Precision@20: 0.250
Precision@50: 0.420
Baseline Precision@10: 0.600
Baseline Precision@20: 0.600
Baseline Precision@50: 0.500


## Model comparison

| Metric | Week-4 baseline | Week-5 logistic regression |
|---|---:|---:|
| Precision@10 | 0.600 | 0.400 |
| Precision@20 | 0.600 | 0.250 |
| Precision@50 | 0.500 | 0.420 |

The logistic regression model did not outperform the Week-4 baseline on any of the three ranking metrics. The baseline performed better at Precision@10, Precision@20, and Precision@50.

This suggests that, with these two features, the learned model does not provide a stronger ranking signal than the simple baseline. I therefore treat the Week-4 baseline as the stronger decision-support approach for this dataset and avoid claiming that the Week-5 model improves performance.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model's errors show that the two selected features do not fully separate declining from non-declining content.

The model uses days since last update and impressions over the last 90 days. These features provide useful signals, but they are not sufficient to capture every reason why content may decline.

The baseline performs better in the top-ranked results, suggesting that the direct action-oriented combination of content staleness and impression volume is more useful for prioritization than the logistic regression ranking in this experiment.

The main interpretation is therefore negative but useful: adding a simple learned model did not improve the existing baseline under this evaluation setup. The result supports keeping the baseline as the current decision-support method rather than replacing it with the tested model.

In [35]:
# Inspect the model coefficients
model_coefficients = pd.Series(
    model.named_steps["model"].coef_[0],
    index=features
).sort_values(ascending=False)

print("Model coefficients:")
print(model_coefficients)

# Show the highest-scored test examples
error_view = X_test.copy()
error_view["actual"] = y_test
error_view["model_score"] = test_scores

print("\nTop 10 model-ranked examples:")
display(
    error_view.sort_values("model_score", ascending=False).head(10)
)

Model coefficients:
days_since_last_update    0.179369
impressions_90d          -0.065083
dtype: float64

Top 10 model-ranked examples:


,days_since_last_update,impressions_90d,actual,model_score
26242,373,35,1,0.829777
15608,334,10,0,0.805006
23506,305,10,0,0.784866
26249,305,13,0,0.784864
21201,305,15,0,0.784862
23741,304,206,1,0.784015
7222,301,7,1,0.781974
22248,301,64,0,0.781936
1659,236,75,1,0.731027
14545,211,1,0,0.709622


The top-10 inspection shows that the model correctly ranks 4 of the 10 highest-scored items as declining. The remaining 6 are false positives under the evaluation label. This is consistent with the measured Precision@10 of 0.400.

The model tends to rank content with high values of `days_since_last_update`, but high staleness alone does not guarantee that content is declining. This helps explain why the learned model underperforms the Week-4 baseline for top-ranked recommendations.

In [36]:
print(model_coefficients)

days_since_last_update    0.179369
impressions_90d          -0.065083
dtype: float64


## Self-check

- [x] Every section above is filled with both markdown reasoning and supporting code.
- [x] The notebook uses a reproducible 80/20 stratified train/test split.
- [x] The logistic regression model uses the same two inputs as the Week-4 baseline.
- [x] Precision@10, Precision@20, and Precision@50 were measured on the held-out test set.
- [x] The Week-5 model was compared directly with the Week-4 baseline.
- [x] The Week-5 model did not outperform the Week-4 baseline, and this is reported honestly.
- [x] Error analysis was performed on the model's top-ranked predictions.
- [x] No client names, URLs, or private queries are included.
- [x] Claims are limited to what was observed and measured.
- [x] The notebook runs top-to-bottom with no errors.
- [ ] The completed notebook is committed to the repository under `work/notebooks/`.